In [ ]:
#Download asvspoof-2019-la-subset dataset from kaggle
import kagglehub
path = kagglehub.dataset_download("beosup/asvspoof-2019-la-subset")

100%|██████████| 7.12G/7.12G [03:01<00:00, 42.2MB/s]

Extracting files...


In [ ]:
from pathlib import Path
#convert path string to path object
base_dir = Path(path)

# DATA PREPARATION

### PHASE 1 : METADATA EXTRACTION & LABELING

In [ ]:
#training protocol text file
cm_protocol_train = list(base_dir.rglob("*.train.trn.txt"))[0]

d_meta = {} #dict for mapping ids files with labels
file_ids = [] #list of ids files

with open(cm_protocol_train, "r") as f :
  for line in f :
    _, key, _, _, label = line.strip().split()
    file_ids.append(key)
    # Binary label mapping: 1 bonafide, 0 for spoof
    d_meta[key] = 1 if label == "bonafide" else 0


In [ ]:
# Locate the first .flac file dynamically to identify the parent audio directory
sample_file = list(base_dir.rglob(f"{file_ids[0]}.flac"))[0]
audio_dir = sample_file.parent

### PHASE 2: AUDIO LOADING & DIGITIZATION

In [ ]:
import soundfile as sf
def load_audio (file_id, audio_dir) :
  """
  Constructs the .flac file path and reads the raw waveform.
  Returns the 1D amplitude-time vector and the sampling rate (16 kHz).
  """
  audio_path=audio_dir / f"{file_id}.flac"
  #sf.read returns a tuple : 1D audio array, sample rate
  vector, sample_rate = sf.read(str(audio_path))
  return vector, sample_rate



### DURATION NORMALIZATION (4S)

In [ ]:
import numpy as np
import torch

def pad_or_crop (vector, target_length=64600) :
  """
    Standardizes each audio signal to exactly 64,600 samples (~4 seconds at 16kHz).
    - If shorter: repeats/tiles the signal (np.tile) and crops to 64,600.
    - If longer : truncates directly to the first 64,600 samples.
    Converts the resulting NumPy array into a PyTorch float32 Tensor.
  """
  length=len(vector)

  if length < target_length :
    num_repeats = (target_length // length) +1
    vector = np.tile(vector, num_repeats)[:target_length]

  else :
   vector = vector[:target_length]

  return torch.from_numpy(vector).float()

### PYTORCH CUSTOM DATASET DEFINITION

In [ ]:
from torch.utils.data import Dataset

class Dataset_ASVspoof2019_train(Dataset):
  def __init__(self, file_ids, d_meta, audio_dir):
    self.file_ids = file_ids   # List of file IDs
    self.d_meta = d_meta       # Metadata dictionary {id: label}
    self.audio_dir = audio_dir # Path to directory containing .flac files


  def __len__ (self):
    """Returns the total number of audio samples in the dataset."""
    return len(self.file_ids)

  def __getitem__(self, index):
    """
        Fetches a single sample at a given index:
        1. Retrieves file key and binary label (converted to torch.long).
        2. Loads raw audio waveform using load_audio.
        3. Normalizes duration to 4 seconds (64,000 samples) via pad_or_crop.
        4. Returns the tuple (normalized_tensor, label_tensor).
    """
    key = self.file_ids[index]
    label = self.d_meta[key]
    label = torch.tensor(label, dtype=torch.long)

    vector, _ = load_audio (key, self.audio_dir)
    vector = pad_or_crop (vector)

    return vector, label

### TEST

In [ ]:
from torch.utils.data import DataLoader
train_dataset = Dataset_ASVspoof2019_train(
    file_ids=file_ids,
    d_meta=d_meta,
    audio_dir=audio_dir
)

# DataLoader (batch size = 32)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# Test of the 1st batch
x_batch, y_batch = next(iter(train_loader))

print("Data Pipeline operationnel ")
print("batch audio (X) format :", x_batch.shape)  # Affiche torch.Size([32, 64600])
print("batch labels (Y) format :", y_batch.shape) # Affiche torch.Size([32])

Data Pipeline operationnel 
batch audio (X) format : torch.Size([32, 64600])
batch labels (Y) format : torch.Size([32])
